In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from lightgbm import LGBMClassifier
import os
import sklearn

In [2]:
sklearn.set_config(transform_output="pandas")

In [3]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)

In [5]:
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

/tmp/ipykernel_10407/3837567942.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [6]:
# 1. Imputation
imputer = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numerical_cols),
        ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

In [7]:
# 2. Feature Engineering with Clustering and Interactions
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
        self.scaler = StandardScaler()
        self.cluster_features = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours']
        
    def fit(self, X, y=None):
        X_cluster = X[self.cluster_features].copy()
        X_cluster_scaled = self.scaler.fit_transform(X_cluster)
        self.kmeans.fit(X_cluster_scaled)
        return self
    
    def transform(self, X):
        X_out = X.copy()
        
        # Interactions
        denom_screen = X_out['daily_screen_time_hours'].replace(0, 0.001)
        denom_notif = X_out['notifications_per_day'].replace(0, 0.001)
        
        X_out['social_media_ratio'] = X_out['social_media_hours'] / denom_screen
        X_out['gaming_ratio'] = X_out['gaming_hours'] / denom_screen
        X_out['work_study_ratio'] = X_out['work_study_hours'] / denom_screen
        X_out['app_opens_per_hour'] = X_out['app_opens_per_day'] / denom_screen
        X_out['notifications_to_opens_ratio'] = X_out['app_opens_per_day'] / denom_notif
        X_out['sleep_deficit'] = 8.0 - X_out['sleep_hours']
        
        # Additional interactions (multiplying features)
        X_out['screen_time_x_notifications'] = X_out['daily_screen_time_hours'] * X_out['notifications_per_day']
        X_out['social_media_x_app_opens'] = X_out['social_media_hours'] * X_out['app_opens_per_day']
        
        # Clustering
        X_cluster = X_out[self.cluster_features].copy()
        X_cluster_scaled = self.scaler.transform(X_cluster)
        X_out['behavior_cluster_id'] = self.kmeans.predict(X_cluster_scaled)
        
        return X_out

In [8]:
# 3. Final Preprocessing
final_preprocessor = ColumnTransformer(
    transformers=[
        ('cat_encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

In [9]:
preprocessor = Pipeline(steps=[
    ('imputer', imputer),
    ('engineer', FeatureEngineer()),
    ('final_preprocessor', final_preprocessor)
])

In [10]:
print("Preprocessing data...")
X_preprocessed = preprocessor.fit_transform(X)
X_test_preprocessed = preprocessor.transform(X_test)

Preprocessing data...


In [11]:
# Convert categorical and new cluster ID to category dtype
for col in categorical_cols:
    X_preprocessed[col] = X_preprocessed[col].astype('category')
    X_test_preprocessed[col] = X_test_preprocessed[col].astype('category')
X_preprocessed['behavior_cluster_id'] = X_preprocessed['behavior_cluster_id'].astype('category')
X_test_preprocessed['behavior_cluster_id'] = X_test_preprocessed['behavior_cluster_id'].astype('category')

In [12]:
print("Training model...")
model = LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
model.fit(X_preprocessed, y)

Training model...


,learning_rate,0.05
,n_estimators,200
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [13]:
print("Predicting final results...")
final_preds = model.predict_proba(X_test_preprocessed)[:, 1]

Predicting final results...


In [14]:
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_preds})
submission.to_csv('submissions/deeper_features.csv', index=False)
print("Submission saved to submissions/deeper_features.csv")

Submission saved to submissions/deeper_features.csv
